In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [8]:
df = pd.read_excel("data/raw/Aggregated_Flood_Data.xlsx", sheet_name="Aggregated")

start = pd.to_datetime(df["Start Date"], errors="coerce")
end = pd.to_datetime(df["End Date"], errors="coerce")
df["Duration"] = (end - start).dt.days.clip(lower=0)

df = df[df["Year"].between(1900, 2026)].dropna(subset=["Fatalities"])

numeric_features = ["Year", "Lat", "Lon", "Duration", "Affected", "Injured"]
categorical_features = ["Country"]

X = df[numeric_features + categorical_features]
y = (df["Fatalities"] >= 1).astype(int)

X.shape, y.value_counts(normalize=True)

((129721, 7),
 Fatalities
 0    0.82837
 1    0.17163
 Name: proportion, dtype: float64)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=522
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(103776, 7) (25945, 7)
0.1716 0.1716


In [10]:
preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical_features),
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

X_train_proc.shape, X_test_proc.shape

((103776, 377), (25945, 377))

## Model Selection

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=522)
scoring = ["f1", "recall"]

In [17]:
dummy_pipe = make_pipeline(preprocessor, DummyClassifier(strategy="most_frequent"))
dummy_scores = cross_validate(dummy_pipe, X_train, y_train, cv=cv, scoring=scoring)

results = {"Dummy": {m: dummy_scores[f"test_{m}"].mean() for m in scoring}}
pd.DataFrame(results).T.round(4)

,f1,recall
Dummy,0.0,0.0


In [20]:
models = {
    "Logistic Regression": make_pipeline(preprocessor, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=522)),
    "Random Forest": make_pipeline(preprocessor, RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", n_jobs=-1, random_state=522)),
}

for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {m: scores[f"test_{m}"].mean() for m in scoring}